# Gold: physical_vendas_caixa
Lê a Silver e cria as tabelas Gold com as **regras de negócio (6 a 10)** da planilha `regras.xlsx`,
já agregadas e prontas para o Looker.

Cada tabela é gravada em dois lugares:
1. **Delta** na camada gold do Data Lake (`gold/<tabela>`), com **MERGE (upsert)** pela chave da tabela
2. **SQL Server** (schema `gold` por padrão), espelhando o resultado do MERGE (overwrite + truncate)

| Tabela | Regra | Grão (chave do MERGE) |
|---|---|---|
| `gold_receita_loja_mes` | 6. Receita por loja/mês + MoM | id_loja, ano, mes |
| `gold_ticket_medio_loja_pagamento_ano` | 7. Ticket médio por loja e pagamento/ano | id_loja, tipo_pagamento, ano |
| `gold_cpf_identificado_loja_mes` | 8. % vendas com/sem CPF por loja/mês | id_loja, ano, mes |
| `gold_transacoes_caixa_dia` | 9. Transações por caixa/dia + flag de sobrecarga (+ feriado) | id_loja, id_caixa, data_venda |
| `gold_vendas_dia_feriado` | 10. Vendas por dia com flag é_feriado | data_venda |
| `gold_dim_lojas` | Dimensão de lojas (cidade etc.) para o Looker cruzar com os KPIs | id_loja |

In [0]:
%pip install python-dotenv holidays azure-identity azure-storage-file-datalake --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
from functools import reduce
from dotenv import load_dotenv, find_dotenv
import holidays
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================
# 1. CONFIGURAÇÃO
# =========================================
load_dotenv(find_dotenv(), override=True)

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
ACCOUNT = os.getenv("STORAGE_ACCOUNT_NAME")
RAW_CONTAINER = os.getenv("ADLS_CONTAINER", "raw")
BRONZE_CONTAINER = os.getenv("ADLS_BRONZE_CONTAINER", RAW_CONTAINER)
SILVER_CONTAINER = os.getenv("ADLS_SILVER_CONTAINER", BRONZE_CONTAINER)
GOLD_CONTAINER = os.getenv("ADLS_GOLD_CONTAINER", SILVER_CONTAINER)

# SQL Server
SQL_HOST = os.getenv("SQL_HOST")              # ex.: meuservidor.database.windows.net
SQL_PORT = os.getenv("SQL_PORT", "1433")
SQL_DB = os.getenv("SQL_DATABASE")
SQL_USER = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")
SQL_SCHEMA = os.getenv("SQL_SCHEMA", "gold")
SQL_TRUST_CERT = os.getenv("SQL_TRUST_CERT", "false")  # "true" só para servidor sem certificado válido

pendentes = [n for n, v in {
    "ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID, "ADLS_CLIENT_SECRET": CLIENT_SECRET,
    "STORAGE_ACCOUNT_NAME": ACCOUNT, "SQL_HOST": SQL_HOST, "SQL_DATABASE": SQL_DB,
    "SQL_USERNAME": SQL_USER, "SQL_PASSWORD": SQL_PASSWORD,
}.items() if not v]
if pendentes:
    raise ValueError(f"Variáveis pendentes no .env: {pendentes}")

host = f"{ACCOUNT}.dfs.core.windows.net"
spark_auth = {
    f"fs.azure.account.auth.type.{host}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{host}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{host}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{host}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{host}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

# SDK do Azure só para listar pastas/arquivos (o cluster não aceita spark.conf.set de credenciais)
adls = DataLakeServiceClient(
    account_url=f"https://{host}",
    credential=ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET),
)

def abfss(container, rel):
    return f"abfss://{container}@{host}/{rel.strip('/')}"

SILVER_PATH = abfss(SILVER_CONTAINER, "silver/physical_vendas_caixa")
LOJAS_REL = os.getenv("LOJAS_REL")  # opcional: caminho de physical_lojas dentro do container raw

# No serverless, o Spark só grava no SQL Server pelo conector nativo "sqlserver" (o "jdbc" genérico é bloqueado)
sqlserver_opts = {
    "host": SQL_HOST,
    "port": SQL_PORT,
    "database": SQL_DB,
    "user": SQL_USER,
    "password": SQL_PASSWORD,
    "encrypt": "true",
    "trustServerCertificate": SQL_TRUST_CERT,
}

# Regra 9: caixa é "sobrecarregado" quando faz X vezes mais transações que a média do dia
FATOR_SOBRECARGA = 3.0

In [0]:
# =========================================
# 2. FUNÇÕES DE APOIO
# =========================================
def listar(container, rel="", recursivo=False):
    fs = adls.get_file_system_client(container)
    try:
        return [(p.name, p.is_directory) for p in fs.get_paths(path=rel or None, recursive=recursivo)]
    except Exception:
        return []

def delta_existe(container, rel):
    return any("_delta_log" in n for n, _ in listar(container, rel))

def ler_delta(path, nome):
    try:
        return spark.read.format("delta").options(**spark_auth).load(path)
    except Exception as e:
        raise RuntimeError(f"Não foi possível ler {nome} em {path}. Detalhe: {e}")

# ---------- Delta: MERGE (upsert) ----------
def merge_delta(df, tabela, chaves):
    rel = f"gold/{tabela}"
    path = abfss(GOLD_CONTAINER, rel)

    if not delta_existe(GOLD_CONTAINER, rel):
        df.write.format("delta").mode("overwrite").options(**spark_auth).save(path)
        print(f"  [Delta] {tabela}: criada ({df.count()} linhas)")
        return path

    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in chaves)
    try:
        from delta.tables import DeltaTable
        destino = DeltaTable.forPath(spark, path, spark_auth)
        (destino.alias("t").merge(df.alias("s"), cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"  [Delta] {tabela}: MERGE concluído")
    except Exception as e:
        # Plano B (mesmo resultado de um upsert): mantém as linhas antigas cujas chaves não vieram
        # na carga nova e substitui/inclui as que vieram. O Delta permite ler e sobrescrever o mesmo caminho.
        print(f"  [Delta] {tabela}: MERGE via API indisponível ({type(e).__name__}); aplicando upsert equivalente")
        atual = spark.read.format("delta").options(**spark_auth).load(path)
        mantidas = atual.join(df.select(*chaves), on=chaves, how="left_anti")
        (mantidas.unionByName(df, allowMissingColumns=True)
            .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .options(**spark_auth).save(path))
    return path

# ---------- SQL Server ----------
# O MERGE (upsert) acontece na tabela Delta da gold. Depois, o resultado completo do MERGE é
# espelhado no SQL Server com overwrite + truncate (mantém a estrutura da tabela, troca as linhas).
# Resultado no SQL Server = exatamente o da tabela Delta após o upsert. Tudo via conector
# "sqlserver" do Spark, que é o único permitido no serverless (sem drivers Python que travam o kernel).
def ler_sqlserver(consulta):
    return (spark.read.format("sqlserver").options(**sqlserver_opts)
            .option("dbtable", f"({consulta}) AS q").load())

def sync_sqlserver(path_delta, tabela):
    df_final = spark.read.format("delta").options(**spark_auth).load(path_delta)
    (df_final.write.format("sqlserver").options(**sqlserver_opts)
        .option("dbtable", f"{SQL_SCHEMA}.{tabela}")
        .option("truncate", "true")
        .mode("overwrite").save())
    print(f"  [SQL Server] {SQL_SCHEMA}.{tabela}: sincronizada com a Delta")

# Teste de conexão logo no início, para falhar cedo se o SQL Server não estiver acessível
teste = ler_sqlserver(f"SELECT DB_NAME() AS banco, SCHEMA_ID(N'{SQL_SCHEMA}') AS schema_id").first()
print(f"SQL Server conectado: banco={teste['banco']} | schema '{SQL_SCHEMA}' existe: {teste['schema_id'] is not None}")
if teste["schema_id"] is None:
    raise RuntimeError(f"O schema '{SQL_SCHEMA}' não existe no banco. Peça para criar (CREATE SCHEMA {SQL_SCHEMA}) "
                       f"ou use SQL_SCHEMA=dbo no .env.")

def publicar(df, tabela, chaves):
    df = df.withColumn("gold_updated_at", F.current_timestamp())
    print(f"\n{tabela}  (chave: {', '.join(chaves)})")
    path = merge_delta(df, tabela, chaves)
    sync_sqlserver(path, tabela)

SQL Server conectado: banco=internshipDatabase | schema 'squad3' existe: True


In [0]:
# =========================================
# 3. LEITURA DA SILVER + COLUNAS DE APOIO
# =========================================
silver = ler_delta(SILVER_PATH, "a Silver")

# CPF identificado = CPF com 11 dígitos (ignora nulo, vazio e o texto "null")
cpf_digitos = F.regexp_replace(F.coalesce(F.col("cpf_cliente"), F.lit("")), "[^0-9]", "")

vendas = (
    silver
    .withColumn("data_venda", F.to_date("dt_venda"))
    .withColumn("cpf_identificado", F.length(cpf_digitos) == 11)
)
# (sem .cache(): o compute serverless não permite cache/persist)
print(f"Linhas na Silver: {vendas.count()}")

Linhas na Silver: 375000


In [0]:
# =========================================
# 4. REGRA 6: Receita total por loja por mês (MoM por loja)
# =========================================
w_loja = Window.partitionBy("id_loja").orderBy("periodo")

receita_loja_mes = (
    vendas.groupBy("id_loja", "ano", "mes")
    .agg(
        F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
        F.count("*").alias("qtd_transacoes"),
    )
    .withColumn("periodo", F.make_date("ano", "mes", F.lit(1)))
    .withColumn("_periodo_ant", F.lag("periodo").over(w_loja))
    .withColumn("_receita_ant", F.lag("receita_total").over(w_loja))
    # só compara com o mês imediatamente anterior (se a loja não vendeu no mês anterior, MoM fica nulo)
    .withColumn(
        "receita_mes_anterior",
        F.when(F.col("_periodo_ant") == F.add_months("periodo", -1), F.col("_receita_ant")),
    )
    .withColumn(
        "variacao_mom_pct",
        F.round((F.col("receita_total") - F.col("receita_mes_anterior"))
                / F.col("receita_mes_anterior") * 100, 2),
    )
    .drop("_periodo_ant", "_receita_ant")
)
publicar(receita_loja_mes, "gold_receita_loja_mes", ["id_loja", "ano", "mes"])


gold_receita_loja_mes  (chave: id_loja, ano, mes)
  [Delta] gold_receita_loja_mes: MERGE via API indisponível (UnknownException); aplicando upsert equivalente
  [SQL Server] squad3.gold_receita_loja_mes: sincronizada com a Delta


In [0]:
# =========================================
# 5. REGRA 7: Ticket médio por loja e por tipo de pagamento (anual)
# =========================================
ticket_medio = (
    vendas.groupBy("id_loja", "tipo_pagamento", "ano")
    .agg(
        F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
        F.count("*").alias("qtd_transacoes"),
    )
    .withColumn("ticket_medio", F.round(F.col("receita_total") / F.col("qtd_transacoes"), 2))
)
publicar(ticket_medio, "gold_ticket_medio_loja_pagamento_ano", ["id_loja", "tipo_pagamento", "ano"])


gold_ticket_medio_loja_pagamento_ano  (chave: id_loja, tipo_pagamento, ano)
  [Delta] gold_ticket_medio_loja_pagamento_ano: criada (261 linhas)
  [SQL Server] squad3.gold_ticket_medio_loja_pagamento_ano: sincronizada com a Delta


In [0]:
# =========================================
# 6. REGRA 8: % de vendas com CPF identificado vs sem CPF, por loja por mês
# =========================================
cpf_loja_mes = (
    vendas.groupBy("id_loja", "ano", "mes")
    .agg(
        F.count("*").alias("qtd_transacoes"),
        F.sum(F.col("cpf_identificado").cast("int")).alias("qtd_com_cpf"),
    )
    .withColumn("qtd_sem_cpf", F.col("qtd_transacoes") - F.col("qtd_com_cpf"))
    .withColumn("pct_com_cpf", F.round(F.col("qtd_com_cpf") / F.col("qtd_transacoes") * 100, 2))
    .withColumn("pct_sem_cpf", F.round(F.col("qtd_sem_cpf") / F.col("qtd_transacoes") * 100, 2))
)
publicar(cpf_loja_mes, "gold_cpf_identificado_loja_mes", ["id_loja", "ano", "mes"])


gold_cpf_identificado_loja_mes  (chave: id_loja, ano, mes)
  [Delta] gold_cpf_identificado_loja_mes: criada (928 linhas)
  [SQL Server] squad3.gold_cpf_identificado_loja_mes: sincronizada com a Delta


In [0]:
# =========================================
# 7. REGRA 10: tabela de feriados nacionais (usada nas regras 9 e 10)
# =========================================
anos = [r["ano"] for r in vendas.select("ano").distinct().collect() if r["ano"] is not None]
feriados = spark.createDataFrame(
    [(d, nome) for d, nome in sorted(holidays.Brazil(years=anos).items())],
    "data_feriado DATE, nome_feriado STRING",
)

def com_feriado(df):
    return (
        df.join(F.broadcast(feriados), df["data_venda"] == feriados["data_feriado"], "left")
          .withColumn("e_feriado", F.col("data_feriado").isNotNull())
          .drop("data_feriado")
    )

In [0]:
# =========================================
# 8. REGRA 9: Transações por caixa por dia (caixas sobrecarregados)
# Compara cada caixa com a média de transações por caixa NAQUELE DIA em toda a rede.
# =========================================
caixa_dia = (
    vendas.groupBy("id_loja", "id_caixa", "data_venda")
    .agg(
        F.count("*").alias("qtd_transacoes"),
        F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
    )
    .withColumn(
        "media_transacoes_caixa_dia",
        F.round(F.avg("qtd_transacoes").over(Window.partitionBy("data_venda")), 2),
    )
    .withColumn(
        "razao_vs_media",
        F.round(F.col("qtd_transacoes") / F.col("media_transacoes_caixa_dia"), 2),
    )
    .withColumn("caixa_sobrecarregado", F.col("razao_vs_media") >= F.lit(FATOR_SOBRECARGA))
    .withColumn("ano", F.year("data_venda"))
    .withColumn("mes", F.month("data_venda"))
)
caixa_dia = com_feriado(caixa_dia)
publicar(caixa_dia, "gold_transacoes_caixa_dia", ["id_loja", "id_caixa", "data_venda"])


gold_transacoes_caixa_dia  (chave: id_loja, id_caixa, data_venda)
  [Delta] gold_transacoes_caixa_dia: criada (123631 linhas)
  [SQL Server] squad3.gold_transacoes_caixa_dia: sincronizada com a Delta


In [0]:
# =========================================
# 9. REGRA 10: Vendas por dia com flag de feriado (feriado vs dia normal)
# =========================================
vendas_dia = (
    vendas.groupBy("data_venda")
    .agg(
        F.count("*").alias("qtd_transacoes"),
        F.round(F.sum("valor_total_venda"), 2).alias("receita_total"),
        F.countDistinct("id_loja").alias("qtd_lojas_com_venda"),
    )
    .withColumn("ticket_medio", F.round(F.col("receita_total") / F.col("qtd_transacoes"), 2))
    .withColumn("ano", F.year("data_venda"))
    .withColumn("mes", F.month("data_venda"))
    .withColumn("dia_semana", F.date_format("data_venda", "EEEE"))  # nome em inglês; o Looker pode traduzir
    .withColumn("fim_de_semana", F.dayofweek("data_venda").isin(1, 7))
)
vendas_dia = com_feriado(vendas_dia)
publicar(vendas_dia, "gold_vendas_dia_feriado", ["data_venda"])


gold_vendas_dia_feriado  (chave: data_venda)
  [Delta] gold_vendas_dia_feriado: criada (974 linhas)
  [SQL Server] squad3.gold_vendas_dia_feriado: sincronizada com a Delta


In [0]:
# =========================================
# 10. DIMENSÃO DE LOJAS (para o Looker cruzar KPIs com cidade, nome etc.)
# Lê physical_lojas da raw (mesma lógica da Silver) e publica todas as colunas dela.
# =========================================
def localizar_lojas():
    if LOJAS_REL:
        return LOJAS_REL.strip("/")
    candidatos = [n for n, _ in listar(RAW_CONTAINER, recursivo=True)
                  if n.split("/")[0].lower() not in ("bronze", "silver", "gold") and "_delta_log" not in n]
    for termo in ("physical_lojas", "lojas"):
        achados = [n for n in candidatos if termo in n.split("/")[-1].lower()]
        if achados:
            return min(achados, key=len)
    return None

def ler_auto(container, rel):
    path = abfss(container, rel)
    arquivos = [n for n, d in listar(container, rel, recursivo=True) if not d] or [rel]
    if any("_delta_log" in n for n in arquivos):
        return spark.read.format("delta").options(**spark_auth).load(path)
    exts = {n.rsplit(".", 1)[-1].lower() for n in arquivos if "." in n.split("/")[-1]}
    leitor = spark.read.options(**spark_auth)
    if "parquet" in exts:
        return leitor.parquet(path)
    if "json" in exts:
        return leitor.option("multiLine", "true").json(path)
    for sep in [",", ";", "|"]:
        df_csv = leitor.option("header", "true").option("inferSchema", "true").option("sep", sep).csv(path)
        if len(df_csv.columns) > 1:
            return df_csv
    return df_csv

lojas_rel = localizar_lojas()
if lojas_rel:
    dim_lojas = (
        ler_auto(RAW_CONTAINER, lojas_rel)
        .withColumn("id_loja", F.col("id_loja").cast("int"))
        .dropDuplicates(["id_loja"])
    )
    print(f"Lojas lidas de {RAW_CONTAINER}/{lojas_rel}: colunas {dim_lojas.columns}")
    publicar(dim_lojas, "gold_dim_lojas", ["id_loja"])
else:
    print("AVISO: physical_lojas não encontrada na raw; gold_dim_lojas não foi publicada. Defina LOJAS_REL no .env.")

Lojas lidas de raw/batch-data/physical_lojas.csv: colunas ['id_loja', 'nome_loja', 'cnpj', 'cidade_loja', 'estado_loja', 'peso_vendas']

gold_dim_lojas  (chave: id_loja)
  [Delta] gold_dim_lojas: criada (29 linhas)
  [SQL Server] squad3.gold_dim_lojas: sincronizada com a Delta


In [0]:
# =========================================
# 11. VALIDAÇÃO: contagem no Delta x SQL Server
# =========================================
tabelas = [
    "gold_receita_loja_mes", "gold_ticket_medio_loja_pagamento_ano", "gold_cpf_identificado_loja_mes",
    "gold_transacoes_caixa_dia", "gold_vendas_dia_feriado",
] + (["gold_dim_lojas"] if lojas_rel else [])

print(f"{'tabela':<42}{'delta':>10}{'sql server':>12}")
for t in tabelas:
    n_delta = ler_delta(abfss(GOLD_CONTAINER, f"gold/{t}"), t).count()
    n_sql = ler_sqlserver(f"SELECT COUNT(*) AS n FROM [{SQL_SCHEMA}].[{t}]").first()["n"]
    status = "OK" if n_delta == n_sql else "DIVERGENTE"
    print(f"{t:<42}{n_delta:>10}{n_sql:>12}  {status}")

# Conferência de negócio: a receita total da Gold deve bater com a Silver
receita_silver = vendas.agg(F.round(F.sum("valor_total_venda"), 2)).first()[0]
receita_gold = receita_loja_mes.agg(F.round(F.sum("receita_total"), 2)).first()[0]
print(f"\nReceita Silver: {receita_silver} | Receita Gold (regra 6): {receita_gold}")

print("\nCaixas sobrecarregados (amostra):")
display(caixa_dia.filter("caixa_sobrecarregado").orderBy(F.desc("razao_vs_media")).limit(20))

print("\nFeriado x dia normal:")
display(vendas_dia.groupBy("e_feriado").agg(
    F.count("*").alias("dias"),
    F.round(F.avg("receita_total"), 2).alias("receita_media_dia"),
    F.round(F.avg("ticket_medio"), 2).alias("ticket_medio"),
))

tabela                                         delta  sql server
gold_receita_loja_mes                            928         928  OK
gold_ticket_medio_loja_pagamento_ano             261         261  OK
gold_cpf_identificado_loja_mes                   928         928  OK
gold_transacoes_caixa_dia                     123631      123631  OK
gold_vendas_dia_feriado                          974         974  OK
gold_dim_lojas                                    29          29  OK

Receita Silver: 137214981.41 | Receita Gold (regra 6): 137214981.41

Caixas sobrecarregados (amostra):


id_loja,id_caixa,data_venda,qtd_transacoes,receita_total,media_transacoes_caixa_dia,razao_vs_media,caixa_sobrecarregado,ano,mes,nome_feriado,e_feriado
10,4,2024-04-18,15,5174.73,3.02,4.97,true,2024,4,null,false
6,3,2024-08-29,13,5151.91,2.89,4.5,true,2024,8,null,false
10,5,2026-04-20,13,6033.83,2.89,4.5,true,2026,4,null,false
6,2,2024-09-11,12,3941.78,2.7,4.44,true,2024,9,null,false
10,5,2024-08-24,11,3265.32,2.5,4.4,true,2024,8,null,false
2,4,2025-02-18,10,3262.66,2.29,4.37,true,2025,2,null,false
10,2,2025-01-18,10,3681.0,2.3,4.35,true,2025,1,null,false
7,1,2024-09-24,11,3141.03,2.55,4.31,true,2024,9,null,false
9,2,2024-09-24,11,4008.79,2.55,4.31,true,2024,9,null,false
10,2,2024-09-24,11,3899.67,2.55,4.31,true,2024,9,null,false



Feriado x dia normal:


e_feriado,dias,receita_media_dia,ticket_medio
false,950,140632.47,366.16
true,24,150589.06,360.08
